# Dense exact match evaluation

This notebook checks the efficient blocker-aware evaluator in `liars_poker.eval.match_dense` against the older generic exact evaluator, the exact best-response solver, and direct `NeuralQPolicy` action probabilities.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.infoset import CALL, InfoSet
from liars_poker.policies.tabular_dense import DenseTabularPolicy
from liars_poker.policies.neural_q import (
    NeuralQPolicy,
    compile_neural_q_to_dense,
)
from liars_poker.eval.match import exact_eval_tabular
from liars_poker.eval.match_dense import (
    evaluate_dense_match,
    evaluate_dense_response,
)
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense

In [ ]:
spec = GameSpec(
    ranks=2,
    suits=2,
    hand_size=1,
    claim_kinds=('RankHigh', 'Pair'),
    suit_symmetry=True,
)

def random_dense_policy(seed):
    rng = np.random.default_rng(seed)
    policy = DenseTabularPolicy(spec)
    for hid in range(policy.S.shape[0]):
        cols = np.flatnonzero(policy.legal_mask[hid])
        weights = rng.random((len(policy.hands), len(cols)))
        weights /= weights.sum(axis=1, keepdims=True)
        policy.S[hid] = 0.0
        policy.S[hid][:, cols] = weights
    policy.recompute_likelihoods()
    return policy

print('claims:', len(DenseTabularPolicy(spec).rules.claims))
print('hands:', len(DenseTabularPolicy(spec).hands))

## 1. Compare against the old generic exact evaluator

The policies are stochastic, so this tests probability-weighted responder nodes rather than only deterministic action selection.

In [ ]:
comparison_rows = []
for seed in range(5):
    p1 = random_dense_policy(100 + 2 * seed)
    p2 = random_dense_policy(101 + 2 * seed)

    old_fixed = exact_eval_tabular(spec, p1, p2)
    new_fixed = evaluate_dense_match(spec, p1, p2)
    new_first, new_second = evaluate_dense_response(spec, p2, p1)
    old_swapped = exact_eval_tabular(spec, p2, p1)

    comparison_rows.append({
        'seed': seed,
        'old P1': old_fixed['P1'],
        'new P1': new_fixed['P1'],
        'fixed-seat absolute error': abs(old_fixed['P1'] - new_fixed['P1']),
        'responder-first absolute error': abs(old_fixed['P1'] - new_first),
        'responder-second absolute error': abs(old_swapped['P2'] - new_second),
    })

comparison = pd.DataFrame(comparison_rows).set_index('seed')
display(comparison)
assert comparison.filter(like='absolute error').to_numpy().max() < 1e-7
print('Generic and dense exact evaluators agree.')

## 2. Evaluate an exact best-response policy

Following the policy produced by the exact BR solver must reproduce the solver's role-specific values.

In [ ]:
opponent = random_dense_policy(501)
br_policy, meta = best_response_dense(
    spec,
    opponent,
    debug=False,
    store_state_values=False,
)
expected_first, expected_second = meta['computer'].exploitability()
actual_first, actual_second = evaluate_dense_response(spec, opponent, br_policy)

br_check = pd.Series({
    'solver first': expected_first,
    'evaluator first': actual_first,
    'solver second': expected_second,
    'evaluator second': actual_second,
})
display(br_check)
assert np.allclose(
    (actual_first, actual_second),
    (expected_first, expected_second),
    atol=1e-7,
)
print('Exact BR policy reproduces both exact BR values.')

## 3. Compile a neural Q policy

Every compiled dense row should be deterministic, legal, and choose the same action as `NeuralQPolicy.action_probs`.

In [ ]:
torch.manual_seed(12)
q_policy = NeuralQPolicy(spec, hidden_sizes=(16, 16), device='cpu')
with torch.no_grad():
    for model in (q_policy.model_p1, q_policy.model_p2):
        for parameter in model.parameters():
            parameter.normal_(mean=0.0, std=0.2)
q_policy.eval()

q_dense = compile_neural_q_to_dense(q_policy, batch_size=32)
assert np.allclose(q_dense.S.sum(axis=2), 1.0)
assert np.all((q_dense.S == 0.0) | (q_dense.S == 1.0))
assert not np.any(q_dense.S[~np.broadcast_to(
    q_dense.legal_mask[:, None, :], q_dense.S.shape
)])

for hid in range(q_dense.S.shape[0]):
    history = tuple(
        claim for claim in range(q_dense.k) if (hid >> claim) & 1
    )
    pid = q_dense.pid_to_act(hid)
    for hand_idx, hand in enumerate(q_dense.hands):
        probs = q_policy.action_probs(InfoSet(pid=pid, hand=hand, history=history))
        dense_probs = {
            action: float(q_dense.S[hid, hand_idx, 0 if action == CALL else action + 1])
            for action in q_dense.legal_actions[hid]
        }
        assert probs == dense_probs

print('Neural Q compilation matches direct policy decisions at every infoset.')

## 4. Evaluate the compiled Q policy through both exact paths

In [ ]:
old_q_value = exact_eval_tabular(spec, q_dense, opponent)
new_q_value = evaluate_dense_match(spec, q_dense, opponent)
display(pd.DataFrame([old_q_value, new_q_value], index=['generic exact', 'dense exact']))
assert np.isclose(old_q_value['P1'], new_q_value['P1'], atol=1e-7)
print('All dense match and neural-Q compilation checks passed.')